# Market Making Project: 
## Pricing Calibration
Sections:
* Black-Scholes Pricer
* Black-Scholes IV Inversion
* Heston Pricing Function
* Heston IV Inversion
* Calibration Visualization
* Heston vs. GBM Comparison

Notes:
* We used brentq for root finding and IV inversion. Newtown-Raphson is a faster approach that uses Vega (more common in industry), but brentq is more robust and works well for our smaller dataset
* Using risk-free rate of 3.69% pulled from DGS3MO (FRED) on 4-24-26
* Using just BS inversion we found over 50% of our BS_IV derivations differed from the Barchart IVs by more than 0.5%. To fix this, we've included the Merton (1973) continuous dividend extension
    * q = 4.53% or 0.0453 (pulled from Barchart on 4-29-26)
* Even though TLT options are American, we can still use European pricing methods (Heston) because we're only looking at OTM/ATM options. It is never rational to exercise OTM options, so we can treat them as European 

---
### Black-Scholes Pricer

In [26]:
import numpy as np
def d_calculator(S, K, T, r, q, sigma):
    """ This function creates the d_1 and d_2 values needed for Black-Scholes.
    Args:
        S = Spot price
        K = Strike
        T = Time to maturity (in years)
        r = risk-free rate
        q = annualized dividend yield
        sigma = volatility
    Returns:
        d_1 and d_2 which are components used in BS.
    """
    # Need to check for edge cases, important for calibration later
    if sigma <= 0 or T <= 0:
        return np.nan, np.nan
    
    d_1 = (np.log(S/K) + (r - q + (sigma**2 / 2))*T) / ((sigma * np.sqrt(T)))
    d_2 = d_1 - sigma * np.sqrt(T)

    return d_1, d_2

d_calculator(100, 110, 1, 0.04, 0.0453, 0.15)

(np.float64(-0.5957345320288326), np.float64(-0.7457345320288327))

In [27]:
from scipy.stats import norm
def bs_call(S, K, T, r, q, sigma):
    """ This function prices a call option using the BS formula.
    Args:
        S = Spot price
        K = Strike
        T = Time to maturity (in years)
        r = risk-free rate
        q = annualized dividend yield
        sigma = volatility
    Returns:
        Call price.
    """
    # Use d_calculator() to create d_1 and d_2
    d_1, d_2 = d_calculator(S, K, T, r, q, sigma)

    # Price call using BS formula
    c = (S * np.exp(-q * T) * norm.cdf(d_1)) - (K * np.exp(-r*T) * norm.cdf(d_2))

    return c

bs_call(100, 90, 1, 0.04, 0.0453, 0.15)

np.float64(11.156094881819605)

In [28]:
def bs_put(S, K, T, r, q, sigma):
    """ This function prices a put option using the BS formula.
    Args:
        S = Spot price
        K = Strike
        T = Time to maturity (in years)
        r = risk-free rate
        q = annualized dividend yield
        sigma = volatility
    Returns:
        Put price.
    """
    # Use d_calculator() to create d_1 and d_2
    d_1, d_2 = d_calculator(S, K, T, r, q, sigma)

    # Price put using BS formula
    p = (K * np.exp(-r * T) * norm.cdf(-d_2)) - (S * np.exp(-q * T) * norm.cdf(-d_1))

    return p

bs_put(100, 110, 1, 0.04, 0.0453, 0.15)

np.float64(12.37494550081064)

---
### Black-Scholes IV Inversion

In [29]:
from scipy.optimize import brentq

def bs_iv(market_mid, S, K, T, r, q, flag):
    """
    Inverts BS to find IV given a market mid-price. Uses Brent's method of root finding.
    flag: 'Call' for call, 'Put' for put
    """
    if flag == 'Call':
        pricer = bs_call
    else:
        pricer = bs_put

    # Simple lambda function representing f(sigma) = Price - market_mid
    objective = lambda sigma: pricer(S, K, T, r, q, sigma) - market_mid
    
    try:
        iv = brentq(objective, 1e-4, 5.0)  # search vol between 0.01% and 500%
    except ValueError:
        iv = np.nan  # root not bracketed (bad quote, skip it)
    
    return iv

Now we're ready to use our functions to actually invert BS, find the IVs, and check them against the IVs given in our Barchart data.

In [44]:
import pandas as pd
# Using risk free rate of 3.69%, pulled from FRED (see above notes)
r = 0.0369

# Using spot price of 86.71 (TLT close as of 4/24/26)
S = 86.71

# Using annualized dividend yield of 4.53% (pulled from Barchart)
q = 0.0453
# q = 0.0       # Just to check against Barchart. They likely use q = 0

# Read in cleaned options data
df = pd.read_csv("../data/options/Cleaned Options.csv")

# Calculate BS IVs
df['IV_BS'] = df.apply(
    lambda row: bs_iv(row['Mid'], S, row['Strike'], row['Time to Maturity'], r, q, row['Option Type']),
    axis = 1
)

# Flag entries where Barchart IV and IV_BS differ by >= 0.01 (1%)
df['IV_dif'] = (df['IV_BS'] - df['IV']).abs()
df['Flag'] = (df['IV_dif'] >= 0.01)

print(f"Number of options: {df.shape[0]}")
print(f"Number of options flagged: {df[df['Flag']].shape[0]}")

Number of options: 148
Number of options flagged: 78


IV Validation Summary:
* BS pricer validated against Barchart IVs across 148 OTM options
* Using q = 0 for Barchart comparison (matches their convention)
    * **q = 0.0453 kept in pricer for actual Heston work**
* 90% of options match within 1% absolute IV (mean error 0.56%)
* Residual error grows monotonically with maturity (0.34% at May to 0.81% at Oct)
    * Likely because Barchart uses discrete dividend PV-adjustment rather than continuous yield
    * Yield curve is flat (1MO–6MO spread = 2bp), which rules out rate term structure as a cause
* Residual error is acceptable. We'll use our BS IVs instead of Barcharts
    * **Switched back to the q = 0.0453, so the flags look higher** 

---
### Heston Pricing Function

In [31]:
from scipy.integrate import quad

def heston_cf(u, S, T, r, q, kappa, theta, sigma, rho, v0):
    """
    Heston (1993) characteristic function of ln(S_T) under risk-neutral measure.
    Args:
        u     = integration variable (complex-valued)
        S     = spot price
        T     = time to maturity (in years)
        r     = risk-free rate
        q     = dividend yield
        kappa = mean reversion speed of variance
        theta = long-run variance (long-run vol = sqrt(theta))
        sigma = vol of vol
        rho   = spot-vol correlation (expected negative for TLT)
        v0    = initial variance (current vol = sqrt(v0))
    Returns:
        Complex value of the characteristic function at u
    """
    i = 1j

    # d: analogous to the BS d1/d2 denominator, but complex-valued
    # Captures how fast the variance process mean-reverts under complex rotation
    d = np.sqrt((kappa - rho * sigma * i * u)**2 + sigma**2 * (u**2 + i * u))

    # g: ratio used to simplify the time-dependent terms below
    g = (kappa - rho * sigma * i * u - d) / (kappa - rho * sigma * i * u + d)

    exp_dT = np.exp(-d * T)

    # C: contribution from the drift and long-run variance (kappa, theta)
    C = (r - q) * i * u * T + (kappa * theta / sigma**2) * (
        (kappa - rho * sigma * i * u - d) * T
        - 2 * np.log((1 - g * exp_dT) / (1 - g))
    )

    # D: contribution from the initial variance v0
    D = ((kappa - rho * sigma * i * u - d) / sigma**2) * (
        (1 - exp_dT) / (1 - g * exp_dT)
    )

    return np.exp(C + D * v0 + i * u * np.log(S))

In [36]:
def heston_call(S, K, T, r, q, kappa, theta, sigma, rho, v0):
    """
    Prices a European call using the Heston (1993) model via 
    characteristic function inversion.
    Args:
        S     = spot price
        K     = strike price
        T     = time to maturity (in years)
        r     = risk-free rate
        q     = dividend yield
        kappa = mean reversion speed of variance
        theta = long-run variance
        sigma = vol of vol
        rho   = spot-vol correlation
        v0    = initial variance
    Returns:
        Call price
    """
    if T <= 0:
        return max(S * np.exp(-q * T) - K, 0.0)

    # Forward price: used to normalize the P1 integrand
    F = S * np.exp((r - q) * T)

    # P2: risk-neutral probability that S_T > K
    # Uses standard CF: phi(u)
    integrand_P2 = lambda u: np.real(
        np.exp(-1j * u * np.log(K))
        * heston_cf(u, S, T, r, q, kappa, theta, sigma, rho, v0)
        / (1j * u)
    )

    # P1: stock-measure probability that S_T > K
    # Uses CF shifted by -i: phi(u - i), normalized by forward F
    # The shift by -i changes the measure from risk-neutral to stock measure
    integrand_P1 = lambda u: np.real(
        np.exp(-1j * u * np.log(K))
        * heston_cf(u - 1j, S, T, r, q, kappa, theta, sigma, rho, v0)
        / (1j * u * F)
    )

    P1 = 0.5 + (1 / np.pi) * quad(integrand_P1, 1e-9, 100, limit=200, epsabs=1e-6, epsrel=1e-6)[0]
    P2 = 0.5 + (1 / np.pi) * quad(integrand_P2, 1e-9, 100, limit=200, epsabs=1e-6, epsrel=1e-6)[0]

    return S * np.exp(-q * T) * P1 - K * np.exp(-r * T) * P2

In [33]:
def heston_put(S, K, T, r, q, kappa, theta, sigma, rho, v0):
    """
    Prices a European put using the Heston (1993) model.
    Derived from heston_call via put-call parity — no additional
    integration needed.

    Args: (same as heston_call)
    Returns:
        Put price
    """
    if T <= 0:
        return max(K * np.exp(-r * T) - S * np.exp(-q * T), 0.0)

    c = heston_call(S, K, T, r, q, kappa, theta, sigma, rho, v0)
    return c - S * np.exp(-q * T) + K * np.exp(-r * T)

In [37]:
# Heston with near-zero vol-of-vol should match BS
S, K, T, r, q = 86.71, 87.0, 0.25, 0.0369, 0.0453
vol = 0.15
params = dict(kappa=2.0, theta=vol**2, sigma=1e-6, rho=0.0, v0=vol**2)

h = heston_call(S, K, T, r, q, **params)
b = bs_call(S, K, T, r, q, vol)
print(f"Heston: {h:.6f} | BS: {b:.6f} | Diff: {abs(h-b):.2e}")

Heston: 2.344556 | BS: 2.344570 | Diff: 1.38e-05


---
### Heston IV Inversion

In [39]:
def heston_iv(market_mid, S, K, T, r, q, flag):
    """
    Inverts Heston to find IV given a market mid-price. Uses Brent's method of root finding.
    flag: 'Call' for call, 'Put' for put
    """
    if flag == 'Call':
        pricer = heston_call
    else:
        pricer = heston_put

    # Simple lambda function representing f(sigma) = Price - market_mid
    objective = lambda sigma: pricer(S, K, T, r, q, sigma) - market_mid
    
    try:
        iv = brentq(objective, 1e-4, 5.0)  # search vol between 0.01% and 500%
    except ValueError:
        iv = np.nan  # root not bracketed (bad quote, skip it)
    
    return iv

In [47]:
from scipy.optimize import minimize

def calibration_objective(params, df, S, r, q):
    """
    Given Heston parameters, prices all options, converts to IVs,
    and returns SSE against market IVs.
    
    Args:
        params = [kappa, theta, sigma, rho, v0] (what optimizer adjusts)
        df     = calibration dataframe with market IVs
        S, r, q = spot, rate, dividend yield
    Returns:
        SSE between Heston IVs and market IVs
    """
    kappa, theta, sigma, rho, v0 = params

    sse = 0.0
    for _, row in df.iterrows():
        # Price option using Heston
        if row['Option Type'] == 'Call':
            price = heston_call(S, row['Strike'], row['Time to Maturity'], r, q, kappa, theta, sigma, rho, v0)
        else:
            price = heston_put(S, row['Strike'], row['Time to Maturity'], r, q, kappa, theta, sigma, rho, v0)

        # Convert Heston price to IV using existing bs_iv function
        model_iv = bs_iv(price, S, row['Strike'], row['Time to Maturity'], r, q, row['Option Type'])

        # Accumulate squared error against your market IV
        if not np.isnan(model_iv):
            sse += (model_iv - row['IV_BS'])**2

    return sse

Parameter bounds: (min, max) for each of the 5 parameters
* kappa: mean reversion speed. Must be positive, typically 0-10
* theta: long-run variance. Must be positive. sqrt(theta) = long-run vol
* sigma: vol of vol. Must be positive, typically 0-2
* rho:   spot-vol correlation. Must be in (-1, 1). Expect negative for TLT
* v0:    initial variance. Must be positive. sqrt(v0) should be near ATM IV

In [48]:
bounds = [
    (0.01, 10.0),   # kappa
    (0.001, 0.5),   # theta  (vol range: sqrt(0.001)=3.2% to sqrt(0.5)=70.7%)
    (0.01, 2.0),    # sigma
    (-0.99, 0.99),  # rho
    (0.001, 0.5),   # v0
]

# --- Initial guess ---
# Start near ATM IV (~10-11%) for vol parameters
# v0 and theta as variance: 0.11^2 ≈ 0.012
initial_guess = [1.0, 0.012, 0.3, -0.3, 0.012]

# --- Run calibration ---
result = minimize(
    calibration_objective,
    initial_guess,
    args=(df_calib, S, r, q),
    method='L-BFGS-B',      # gradient-based optimizer that respects bounds
    bounds=bounds,
    options={'maxiter': 1000, 'ftol': 1e-9}
)

kappa_cal, theta_cal, sigma_cal, rho_cal, v0_cal = result.x

print(f"Calibration successful: {result.success}")
print(f"Final SSE: {result.fun:.6f}")
print(f"kappa : {kappa_cal:.4f}  (mean reversion speed)")
print(f"theta : {theta_cal:.4f}  → long-run vol = {np.sqrt(theta_cal)*100:.2f}%")
print(f"sigma : {sigma_cal:.4f}  (vol of vol)")
print(f"rho   : {rho_cal:.4f}  (spot-vol correlation)")
print(f"v0    : {v0_cal:.4f}  → initial vol = {np.sqrt(v0_cal)*100:.2f}%")

Calibration successful: True
Final SSE: 0.007426
kappa : 4.5704  (mean reversion speed)
theta : 0.0223  → long-run vol = 14.94%
sigma : 0.9920  (vol of vol)
rho   : -0.1925  (spot-vol correlation)
v0    : 0.0118  → initial vol = 10.88%


* Kappa = Mean Reversion Speed (how quickly variance snaps back to theta after a shock)
* Theta = Long-Run Vol (after taking sqrt. Since vol is ~15% and current ATM vol is ~11%, we'd say that the market is pricing in mean reversino upwards)
* Sigma = Vol-of-Vol (how much variance itself flucuates. Roughly 1.0 is somewhat high, but this is what's leading to those steep wings in the vol skew)
    * Feller Condition: $ 2 * \kappa * \theta \geq \sigma ^2$, in our case this is being violated, which means that in theory variance can touch zero, which could cause numerical instability during simulations.
        * Commonly violated in practice, but something to note and look out for
* Rho = Spot-Vol Correlation (correlation between TLT price moves and vol moves)
    * Negative, which was expected, because as TLT drops (rates spike), vol increases
* v0 = Initial Vol (this is pretty close to the vol of May's ATM strikes)

In [50]:
import json

# Create the data dictionary
calibration_data = {
    "kappa": float(kappa_cal),
    "theta": float(theta_cal),
    "long_run_vol_pct": float(np.sqrt(theta_cal) * 100),
    "sigma": float(sigma_cal),
    "rho": float(rho_cal),
    "v0": float(v0_cal),
    "initial_vol_pct": float(np.sqrt(v0_cal) * 100)
}

# Write to a file
with open('../data/heston_params.json', 'w') as f:
    json.dump(calibration_data, f, indent=4)

print("Data successfully exported to calibration_results.json")

Data successfully exported to calibration_results.json


---
### Calibration Visualization